In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


# ¿Qué es PEFT + LoRA?

En vez de actualizar todos los parámetros 𝜃, LoRA introduce matrices de bajo rango:

```𝑊′ = 𝑊 + 𝐵𝐴```

donde:

- 𝑊 = pesos originales congelados
- 𝐴,𝐵 = matrices pequeñas entrenables
- Solo se entrenan  𝐴,𝐵

Ventaja:

- Mucho menos VRAM
- Entrenamiento rápido
- Exportable luego a GGUF si deseas usarlo en Ollama

## Listar modelos de huggingface

In [2]:
from huggingface_hub import list_models

models = list_models(filter="text-generation", limit=20)

for m in models:
    print(m.modelId)

e:\OPT\MINICONDA\envs\llms-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


zai-org/GLM-5
Nanbeige/Nanbeige4.1-3B
TeichAI/Qwen3-14B-Claude-4.5-Opus-High-Reasoning-Distill-GGUF
LocoreMind/LocoOperator-4B
MiniMaxAI/MiniMax-M2.5
LiquidAI/LFM2-24B-A2B
Qwen/Qwen3-Coder-Next
CohereLabs/tiny-aya-global
unsloth/Qwen3-Coder-Next-GGUF
Fortytwo-Network/Strand-Rust-Coder-14B-v1
nvidia/NVIDIA-Nemotron-Nano-9B-v2-Japanese
guidelabs/steerling-8b
LiquidAI/LFM2-24B-A2B-GGUF
salakash/Minimalism
nvidia/Qwen3.5-397B-A17B-NVFP4
unsloth/GLM-4.7-Flash-GGUF
zai-org/GLM-4.7-Flash
inclusionAI/Ling-2.5-1T
jdopensource/JoyAI-LLM-Flash
ruv/ruvltra-claude-code


## filtrado avanzado

In [3]:
# Puedes filtrar por:

models = list_models(
    filter="text-generation",
    search="llama",
    limit=20
)

for m in models:
    print(m.modelId, m.sha)

meta-llama/Llama-3.1-8B-Instruct None
meta-llama/Llama-3.2-3B-Instruct None
meta-llama/Meta-Llama-3-8B None
meta-llama/Llama-3.1-8B None
meta-llama/Llama-3.3-70B-Instruct None
meta-llama/Llama-3.2-1B None
DavidAU/Llama-3.2-8X3B-MOE-Dark-Champion-Instruct-uncensored-abliterated-18.4B-GGUF None
meta-llama/Llama-2-7b None
meta-llama/Llama-2-7b-chat-hf None
meta-llama/Meta-Llama-3-8B-Instruct None
meta-llama/Llama-3.2-1B-Instruct None
meta-llama/Llama-3.2-3B None
deepseek-ai/DeepSeek-R1-Distill-Llama-70B None
DavidAU/Llama3.3-8B-Instruct-Thinking-Claude-4.5-Opus-High-Reasoning None
meta-llama/Llama-3.1-405B None
meta-llama/Llama-2-7b-hf None
meta-llama/LlamaGuard-7b None
meta-llama/Llama-3.1-70B-Instruct None
NousResearch/Hermes-3-Llama-3.1-8B None
unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF None


In [4]:
from huggingface_hub import list_models

models = list_models(
    filter="text-generation",
    search="tinyllama",  # 4b
    limit=20
    )

for m in models:
    print(m.modelId)

TinyLlama/TinyLlama-1.1B-Chat-v1.0
TinyLlama/TinyLlama-1.1B-Chat-v0.6
TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
MaziyarPanahi/TinyLlama-1.1B-Chat-v1.0-GGUF
bilalRahib/TinyLLama-NSFW-Chatbot
yanmyoaung04/tinyllama-cybersecurity-model-v1.0
Rj18/text-to-sql-tinyllama-lora
fxmarty/tiny-llama-fast-tokenizer
seanmor5/tiny-llama-test
Maykeye/TinyLLama-v0
reciprocate/tiny-llama
maryxxx/tiny-llamas-110m-trippy
TinyLlama/TinyLlama-1.1B-step-50K-105b
Xenova/TinyLLama-v0
atorsvn/TinyLlama-1.1B-step-50K-105b-gptq-4bit
moraxgiga/Tiny_llama_fine_tuning
sam2ai/tiny_llama_1.1b_odia_ext
casperhansen/tinyllama-1b-awq
mgoin/TinyLlama-1.1B-step-50K-105b-ONNX
TinyLlama/TinyLlama-1.1B-intermediate-step-240k-503b


## Cargar modelo base pequeño

Para tutorial recomiendo algo ligero:

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype
).to(device)

print("Device:", model.device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 277.20it/s, Materializing param=model.norm.weight]                              


Device: cpu


In [6]:
model.device

device(type='cpu')

## USar el modelo

In [19]:
prompt = "what is the economist?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(response)



economist: an expert in economics who studies and analyzes economic issues.

explanation: the economist is a professional who studies and analyzes economic issues to provide insight and recommendations. They use economic data, models, and other tools to make predictions and recommend policies to improve the economy.


## Configurar LoRA

In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # depende del modelo
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## Dataset mínimo sintético

In [14]:
from datasets import Dataset

data = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an analytical editorial writer. Maintain a formal institutional tone, prioritising interpretive analysis over narrative description."
            },
            {
                "role": "user",
                "content": "Provide an overview of The Economist."
            },
            {
                "role": "assistant",
                "content": """## Overview
The Economist is a British weekly publication covering economics, geopolitics, business, technology and culture.

## Historical Origins
Founded in 1843 by James Wilson to advocate the repeal of the Corn Laws, it evolved into a globally recognised journal of political economy.

## Editorial Orientation
The publication aligns with classical and economic liberalism, promoting free markets, deregulation and globalisation.

## Distinctive Features
Articles are published anonymously, reflecting a collective institutional voice. It is recognised for data journalism and interpretive analysis."""
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an analytical editorial writer. Maintain a formal institutional tone and structured analytical format."
            },
            {
                "role": "user",
                "content": "Analyse the ideological positioning of The Economist."
            },
            {
                "role": "assistant",
                "content": """## Ideological Framework
The publication broadly reflects classical and economic liberalism.

## Policy Preferences
It advocates free trade, market deregulation and centrist governance.

## Audience and Influence
Its subscription model and analytical depth position it among business leaders and policy-makers.

## Criticism
Its elite readership and interpretive emphasis have attracted both acclaim and critique."""
            }
        ]
    }
]

dataset = Dataset.from_list(data)

#### Tokenización:

In [18]:
from transformers import DataCollatorForLanguageModeling

def tokenize(example):
    text = "\n".join(f"{m['role']}: {m['content']}" for m in example["messages"])
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_dataset = dataset.map(tokenize, remove_columns=["messages"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Map: 100%|██████████| 2/2 [00:00<00:00, 500.16 examples/s]


#### Entrenamiento

In [20]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    remove_unused_columns=False,
    report_to="none"
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss


Step,Training Loss
1,2.573390
2,2.543656
3,2.522052


TrainOutput(global_step=3, training_loss=2.546366055806478, metrics={'train_runtime': 22.1515, 'train_samples_per_second': 0.271, 'train_steps_per_second': 0.135, 'total_flos': 9544447033344.0, 'train_loss': 2.546366055806478, 'epoch': 3.0})

#### Guardar adaptador LoRA

In [21]:
model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")

('./lora-adapter\\tokenizer_config.json',
 './lora-adapter\\chat_template.jinja',
 './lora-adapter\\tokenizer.json')

# Cargar nuevamente el modelo fine-tuneado (LoRA) en Transformers

- base_model = pesos originales
- PeftModel aplica las matrices LoRA encima

In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./lora-adapter")

model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 973.09it/s, Materializing param=model.norm.weight]                               


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 973.09it/s, Materializing param=model.norm.weight]                               


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_featu

## Fusionar LoRA con el modelo base (recomendado antes de exportar)

Si quieres un modelo independiente (sin necesitar PEFT en inferencia):

In [29]:
prompt ="Explain what The Economist is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(response)



The Economist is an international weekly newspaper published in London, England. It covers news, politics, economics, business, science, technology, and global events. The Economist was founded in 1843 and is published in both print and digital formats. The newspaper's mission is to provide informed commentary on global news and events to its readers. It is known for its editorial independence and its commitment to providing accurate and objective news coverage. The Economist's website, economist.com, provides more in-depth analysis and commentary on current events and issues.


In [30]:
model = model.merge_and_unload()
model.save_pretrained("./merged-model")
tokenizer.save_pretrained("./merged-model")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]


('./merged-model\\tokenizer_config.json',
 './merged-model\\chat_template.jinja',
 './merged-model\\tokenizer.json')